In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import seaborn as sns
from scipy.stats import mannwhitneyu
from cobra.io import read_sbml_model


# Get the current working directory
current_directory = os.getcwd()
# If required go to repository root
if os.path.split(current_directory)[1] != 'PAM_Parametrization':
    # Go up two levels
    parent_directory = os.path.dirname(os.path.dirname(current_directory))
    # Change the directory to the parent directory
    os.chdir(parent_directory)
    
from PAModelpy.utils import set_up_pam
from Scripts.pam_generation import setup_ecoli_pam as set_up_ecoli_pam_curated
from Modules.utils.sector_config_functions import get_translational_sector_config
from Modules.utils.pamparametrizer_analysis import (get_results_from_simulations,
                                                   calculate_error_for_reactions, calculate_r_squared_for_reaction,
                                                   calculate_difference_simulation_experiment)
from Modules.utils.pam_generation import create_pamodel_from_diagnostics_file

# from Modules.utils import calculate_r_squared_for_reaction
# from Scripts.Visualization.PAMparametrizer_progress_cleaned_figure import run_simulations

N_ALT_MODELS = 10
    
ECOLI_PHENOTYPE_DATA_PATH = os.path.join('Data', 'Ecoli_phenotypes')
MODEL_FILE_PATH = os.path.join('Models', 'iML1515.xml')

PARAM_FILE_OLD = os.path.join('Results', '1_preprocessing','proteinAllocationModel_iML1515_EnzymaticData_250912.xlsx')
PARAM_FILE_SCALED = os.path.join('Results', '2_parametrization','proteinAllocationModel_iML1515_EnzymaticData_multi.xlsx')

BEST_INDIV_RESULT_FILES = [os.path.join('Results','2_parametrization','diagnostics',
                                     f'pam_parametrizer_diagnostics_{i}.xlsx') for i in range(1,N_ALT_MODELS+1)]

Loading PAModelpy modules version 0.5.2
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-03


# 1. Load reference data

In [2]:
# load exchange rates for different carbon sources by Gerosa et al. (2015) in Ecoli BW25113
flux_csources = pd.read_excel(os.path.join(ECOLI_PHENOTYPE_DATA_PATH, 'Ecoli_phenotypes_py_rev.xls'),
                       sheet_name = 'Fluxes_Csources',
                            index_col=1)
flux_csources_df = flux_csources.drop(['Flux (publication)', 'Reversibility'], axis=1)
flux_csources_df.head()

,Acetate,Fructose,Galactose,Glucose,Glycerol,Gluconate,Pyruvate,Succinate,Glucose (flux ratio Glc)
Reaction identifier,,,,,,,,,
EX_ac_e_b,13.584,-3.32866,-1.968939e-08,-6.827019,-0.597000,-5.003982,-11.91391,-3.320974,-0.70717
EX_fru_e_b,0.000,8.32800,0.000000e+00,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000
EX_gal_e_b,0.000,0.00000,1.969000e+00,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000
EX_glc__D_e_b,0.000,0.00000,0.000000e+00,9.654000,0.000000,0.000000,0.00000,0.000000,1.00000
EX_glyc_e_b,0.000,0.00000,0.000000e+00,0.000000,4.944834,0.000000,0.00000,0.000000,0.00000


# 2. Setup the *Escherichia coli* iML1515 model with new parameters

In [3]:
#setup the model
gem = read_sbml_model(MODEL_FILE_PATH)
ecoli_pam_wt = set_up_pam(PARAM_FILE_OLD, 
                          model = MODEL_FILE_PATH, 
                          sensitivity=False) # not curation for reference
ecoli_pam_curated = set_up_ecoli_pam_curated(
    pam_data_file_path = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),
    sensitivity = False) # curated for reference

pam = set_up_pam(PARAM_FILE_SCALED, 
                 model = MODEL_FILE_PATH,
                 sensitivity = False)
pam.change_reaction_bounds('EX_glc__D_e', lower_bound=0)

new_ecoli_pams = {alt+1: create_pamodel_from_diagnostics_file(file,
                                          pam.copy(copy_with_pickle=True)) for alt, file in enumerate(BEST_INDIV_RESULT_FILES)}

No enzyme information found for reaction: ICHORS_copy2
No enzyme information found for reaction: ICHORS_copy1
No enzyme information found for reaction: SUCFUMtpp
No enzyme information found for reaction: THMDt2pp_copy1
No enzyme information found for reaction: THMDt2pp_copy2
No enzyme information found for reaction: URIt2pp_copy1
No enzyme information found for reaction: URIt2pp_copy2
No enzyme information found for reaction: ADNt2pp_copy1
No enzyme information found for reaction: ADNt2pp_copy2
No enzyme information found for reaction: GUAtpp
No enzyme information found for reaction: HYXNtpp
No enzyme information found for reaction: FUCtpp
No enzyme information found for reaction: CYTDt2pp_copy1
No enzyme information found for reaction: CYTDt2pp_copy2
No enzyme information found for reaction: CADVtpp
No enzyme information found for reaction: FEENTERtpp
No enzyme information found for reaction: FORtppi
No enzyme information found for reaction: RMNtpp
No enzyme information found for reac

/home/samiralvdb/Documents/3_Projects/7_MCA_analysis/PAModelpy/src/PAModelpy/PAModel.py:252: UserWarning: Molar mass for E332 is invalid: 0.0
  warnings.warn(f"Molar mass for {enz.id} is invalid: {molmass}")


Add the following protein sector: TranslationalProteinSector

Add the following protein sector: UnusedProteinSector

Done with setting up the proteome allocation model iML1515

No enzyme information found for reaction: ICHORS_copy2
No enzyme information found for reaction: ICHORS_copy1
No enzyme information found for reaction: SUCFUMtpp
No enzyme information found for reaction: THMDt2pp_copy1
No enzyme information found for reaction: THMDt2pp_copy2
No enzyme information found for reaction: URIt2pp_copy1
No enzyme information found for reaction: URIt2pp_copy2
No enzyme information found for reaction: ADNt2pp_copy1
No enzyme information found for reaction: ADNt2pp_copy2
No enzyme information found for reaction: GUAtpp
No enzyme information found for reaction: HYXNtpp
No enzyme information found for reaction: FUCtpp
No enzyme information found for reaction: CYTDt2pp_copy1
No enzyme information found for reaction: CYTDt2pp_copy2
No enzyme information found for reaction: CADVtpp
No enzyme i

# 3. Check internal flux distribution

## 3.1 Parse dataframes to validate the flux distribution

In [4]:
# Get the data for growth on multiple carbon sources from Gerosa et al. (2015)
# Make sure all the fluxes of backward reactions are inverted to match the model directionality
fluxes_to_simulate = flux_csources_df.copy()
new_indices = []
for i, row in fluxes_to_simulate.iterrows():
    if isinstance(row.name, str):
        if row.name[-2:] == '_b':
            new_indices.append(row.name[:-2])
            fluxes_to_simulate.loc[row.name]= -row
        else: 
            new_indices.append(row.name)
    else:
        new_indices.append(row.name)
            
fluxes_to_simulate.index = new_indices
fluxes_to_simulate_parsed = fluxes_to_simulate[fluxes_to_simulate.index.notnull()]
fluxes_to_simulate_parsed = fluxes_to_simulate_parsed.rename(
    index = {'BIOMASS_Ec_iML1515_WT_75p37M':'BIOMASS_Ecoli_core_w_GAM'}
).drop('Glucose (flux ratio Glc)', axis = 1)

In [5]:
# extract the validation data and substrate information for each carbon source
flux_mapper = {col: fluxes_to_simulate_parsed.index[i] for i,col in enumerate(fluxes_to_simulate_parsed.columns)}
fluxes_to_save = []
# Get the fluxes we want to save
for flux in fluxes_to_simulate_parsed.index:
    if isinstance(flux, str):
        fluxes_to_save += [f for f in flux.split(', ')]

#parse the fluxes such that we can run and validate simulations easily
validation_df = pd.DataFrame(columns = list(fluxes_to_simulate_parsed.index))
substrate_ids = []
substrate_uptake = []
for substrate, fluxes in fluxes_to_simulate_parsed.items():
    substrate_ids += [flux_mapper[substrate]]
    substrate_uptake += [fluxes.loc[flux_mapper[substrate]]]
    validation_df = pd.concat([validation_df,fluxes.to_frame().T], ignore_index =True)

validation_df.index = list(flux_mapper.values())
validation_df

,EX_ac_e,EX_fru_e,EX_gal_e,EX_glc__D_e,EX_glyc_e,EX_glcn_e,EX_pyr_e,EX_succ_e,EX_fum_e,EX_lac_e,...,TKT2,PPC,PPCK,ICDHyr,SUCDi,FUM,"MDH, MDH2, MDH3","ME1, ME2",ICL,BIOMASS_Ec_iML1515_core_75p37M
EX_ac_e,-1.358400e+01,-0.000,-0.000,-0.000,-0.000000,-0.000000,-0.000,-0.000,1.358425e-07,0.00000,...,0.718249,1.774145,3.114281e+00,4.696417,8.404765,8.404765,10.671300,1.871037e+00,4.137602e+00,0.2900
EX_fru_e,3.328660e+00,-8.328,-0.000,-0.000,-0.000000,-0.000000,-0.000,-0.000,8.327452e-08,0.00000,...,1.788933,3.545669,3.589203e-01,4.566119,3.877105,3.877105,2.227706,1.651610e+00,2.210442e-03,0.4924
EX_gal_e,1.968939e-08,-0.000,-1.969,-0.000,-0.000000,-0.000000,-0.000,-0.000,1.968946e-08,0.00000,...,0.058569,0.377386,8.489495e-01,0.496177,1.260858,1.260858,2.285119,5.237651e-04,1.024785e+00,0.1800
EX_glc__D_e,6.827019e+00,-0.000,-0.000,-9.654,-0.000000,-0.000000,-0.000,-0.000,9.653732e-08,0.00000,...,0.584644,2.453331,5.408740e-01,2.977971,2.138073,2.138073,2.138073,-9.557242e-08,9.654038e-10,0.6500
EX_glyc_e,5.970003e-01,-0.000,-0.000,-0.000,-4.944834,-0.000000,-0.000,-0.000,1.013568e-07,0.00000,...,0.904380,1.376469,-1.003518e-07,2.464836,1.840459,1.840459,1.840459,-1.003463e-07,1.013606e-09,0.4900
EX_glcn_e,5.003982e+00,-0.000,-0.000,-0.000,-0.000000,-7.283923,-0.000,-0.000,7.283677e-08,0.00000,...,0.170985,1.943350,-7.211081e-08,1.154034,0.182358,0.182358,0.182358,-7.210584e-08,7.284001e-08,0.5900
EX_pyr_e,1.191391e+01,-0.000,-0.000,-0.000,-0.000000,-0.000000,-26.714,-0.000,0.000000e+00,1.15701,...,0.090131,2.489449,1.164803e+00,7.979320,7.518256,7.518256,7.427797,1.930607e-01,1.026014e-01,0.3900
EX_succ_e,3.320974e+00,-0.000,-0.000,-0.000,-0.000000,-0.000000,-0.000,-15.902,1.139998e+00,0.00000,...,1.369847,2.016110,2.971789e+00,3.038949,18.370434,17.230436,4.950705,1.239618e+01,1.164515e-01,0.5100


## 3.2 Run simulations

In [ ]:
kwargs = {'substrate_ids': list(validation_df.index), 
          'substrate_rates': [[rate] for rate in substrate_uptake],
          'fluxes_to_save' : fluxes_to_save
         }
fluxes = {'iML1515': get_results_from_simulations(gem, **kwargs)['fluxes']}
trans_sector_config = {sub_id: get_translational_sector_config(ecoli_pam_wt,
                                                                   sub_id,
                                                                   np.arange(-4,-1,1)
                                                                  ) for sub_id in validation_df.index
                          }
# for each study, run simulations
for model_label, pam in zip(['Curated', 'GotEnzymes']+list(new_ecoli_pams.keys()),
                            [ecoli_pam_curated, ecoli_pam_wt]+list(new_ecoli_pams.values())
                           ):
    trans_sector_config = {sub_id: get_translational_sector_config(pam,
                                                                   sub_id,
                                                                   np.arange(-4,-1,1)
                                                                  ) for sub_id in validation_df.index
                          }
    
    pam.change_reaction_bounds('EX_glc__D_e', 0, 1e3)
    fluxes[model_label] = get_results_from_simulations(pam, **kwargs)['fluxes']

Running simulations with -13.58 mmol/g_cdw/h of substrate (EX_ac_e) going into the system
Running simulations with -8.33 mmol/g_cdw/h of substrate (EX_fru_e) going into the system
Running simulations with -1.97 mmol/g_cdw/h of substrate (EX_gal_e) going into the system
Running simulations with -9.65 mmol/g_cdw/h of substrate (EX_glc__D_e) going into the system
Running simulations with -4.94 mmol/g_cdw/h of substrate (EX_glyc_e) going into the system


/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Running simulations with -7.28 mmol/g_cdw/h of substrate (EX_glcn_e) going into the system
Running simulations with -26.71 mmol/g_cdw/h of substrate (EX_pyr_e) going into the system


/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Running simulations with -15.9 mmol/g_cdw/h of substrate (EX_succ_e) going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going i

...

Running simulations with -8.33 mmol/g_cdw/h of substrate (EX_fru_e) going into the system


/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Running simulations with -1.97 mmol/g_cdw/h of substrate (EX_gal_e) going into the system


/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Running simulations with -9.65 mmol/g_cdw/h of substrate (EX_glc__D_e) going into the system
Running simulations with -4.94 mmol/g_cdw/h of substrate (EX_glyc_e) going into the system
Running simulations with -7.28 mmol/g_cdw/h of substrate (EX_glcn_e) going into the system
Running simulations with -26.71 mmol/g_cdw/h of substrate (EX_pyr_e) going into the system
Running simulations with -15.9 mmol/g_cdw/h of substrate (EX_succ_e) going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Runn

In [ ]:
error_new_dict = {alt: calculate_error_for_reactions(validation_df,
                                                  fluxes,
                                                  fluxes_to_save[1:]) for alt, fluxes in fluxes.items()}
for alt, error_list in error_new_dict.items():
    print(f'R^2 values for alternative model {alt} with the optimized parameters: ', np.nanmean(error_list))



## 3.2 Visualize the simulation results for the different models

In [ ]:
# validation_df_1.index = validation_df.index.str.split(', ')
validation_df_1 = validation_df.T.reset_index()
validation_df_1['index'] = validation_df_1['index'].str.split(', ')
validation_df_1 = validation_df_1.explode('index').set_index('index').T
validation_df_1
# validation_df = validation_df.explode()

### Boxplots of simulation error

In [ ]:
fontsize = 15
models = [f'Alternative {alt}' for alt in error_new_dict.keys() if not isinstance(alt, str)]
model_colors = sns.color_palette("winter", n_colors=len(models))
cmap = dict(zip(models, model_colors))
cmap = {**cmap,**{'iML1515':'darkblue','GotEnzymes': 'grey', 'Curated': 'chocolate'}}

# Prepare the figure
fig, ax = plt.subplots(figsize=(12, 6))

# Combine data into a DataFrame
all_differences = pd.DataFrame()
curated_differences = None  # Placeholder for Curated errors
num_significant = 0

for col, (model, sub_df) in enumerate(zip(['iML1515','GotEnzymes', 'Curated']+models, fluxes.values())):
    differences = []
    for _, row in sub_df.iterrows():
        substrate_id = row['substrate_id']
        difference = calculate_difference_simulation_experiment(
            validation_df_1, row, fluxes_to_save[1:], substrate_id)
        differences += difference
    
    print(f"{model}: median = {np.median(differences)}, mean = {np.mean(differences)}, stdev = {np.std(differences)}")

    temp_df = pd.DataFrame({'Model': [model] * len(differences), 'Difference': differences})

    # Append to the main DataFrame
    all_differences = pd.concat([all_differences, temp_df], ignore_index=True)

# Boxplot or Violin Plot
sns.boxplot(x='Model', y='Difference', data=all_differences, ax=ax, palette=cmap, showfliers=False,
           )

# Set labels and title
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=fontsize)
ax.set_xlabel('Model', fontsize=fontsize * 1.5)
ax.set_ylabel(r'Difference (exp - sim) mmol/$\text{g}_{CDW}$/h', fontsize=fontsize)
plt.grid()
plt.tight_layout()
# plt.show()
plt.savefig('Figures/SuppFig_multiple_csources_error_boxplot.png')

### line graphs of simulation per reaction

In [ ]:
# visualize per flux
fig, axs = plt.subplots(ncols = 6, nrows = 6, figsize = [20,20])
substrate_ids_cur = fluxes['Curated'].substrate_id
substrate_ids = fluxes['GotEnzymes'].substrate_id


#make the plot panels for each reaction
fig_reactions = []
for i in range(0,36,6):
    fig_reactions += [[rxn for rxn in fluxes_to_save[i:i+6]]]
# plot all reactions
for j in range(6):
    reactions_to_plot = fig_reactions[j]
    for i, rxn in enumerate(reactions_to_plot):
        validation = validation_df_1[rxn]
        axs[j,i].scatter(substrate_ids_cur, validation_df_1.loc[substrate_ids_cur, rxn].abs(), color = 'grey')
#         axs[j,i].plot(substrate_ids_cur, fluxes['Curated'][rxn].abs(), label = 'curated', color = 'black')
#         axs[j,i].plot(fluxes_wt['substrate_id'], fluxes['GotEnzymes'][rxn].abs(),label = 'GotEnzymes', color = 'blue')
        for label, (model, flux_df) in enumerate(zip(['iML1515','GotEnzymes', 'Curated']+models, fluxes.values())):
            axs[j,i].plot(flux_df['substrate_id'], flux_df[rxn].abs(), label = model, 
                          color = cmap[model])

#         axs[j,i].plot(substrate_ids, fluxes_new[rxn].abs(), label = 'PAMparametrizer', color = 'red')
        axs[j,i].set_title(rxn)
        axs[j,i].tick_params(axis='x', labelrotation=60)
        axs[j,i].grid()

        
# Shrink current axis's height by 10% on the bottom
box = axs[j,i].get_position()
axs[j,i].set_position([box.x0, box.y0 + box.height * 0.1,
                 box.width, box.height * 0.9])
        

handles, labels = axs[j,i].get_legend_handles_labels()    
fig.legend(handles, labels, loc = 'lower center', bbox_to_anchor=(0.5, -0.05),ncols = 6, fontsize= fontsize)

fig.supylabel('flux rate [mmol/gCDW/h]', fontsize = fontsize*1.5)
fig.supxlabel('limiting nutrient', fontsize = fontsize*1.5)

plt.subplot_adjust(bottom = 0.5)
plt.tight_layout()

fig.savefig('Figures/SuppFig_fluxgraphs_csources')